# Korean-English 트랜스포머 번역기

## 환경 설정

라이브러리 임포트, 한글 폰트 설정, GPU 인식까지.
RTX 3070 같은 NVIDIA GPU가 있으면 자동으로 CUDA 사용.

In [6]:
import os, re, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# Windows 한글 폰트 설정 (matplotlib 시각화용)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 재현 가능성을 위한 랜덤 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GPU 자동 인식 (CUDA 사용 가능하면 GPU, 아니면 CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device)

torch: 2.5.1+cu121 | device: cuda


## 데이터 경로 설정

이미 다운로드된 Korean-English Park 데이터셋의 경로를 지정.
한국어/영어 파일이 둘 다 `True`로 나오면 성공.

In [7]:
# 이미 다운로드한 데이터 폴더 경로 (Windows 절대 경로)
DATA_DIR = r"C:\AI_study\Transforme\data"

# 한국어/영어 병렬 코퍼스 파일 경로
KOR_PATH = os.path.join(DATA_DIR, "korean-english-park.train.ko")
ENG_PATH = os.path.join(DATA_DIR, "korean-english-park.train.en")

# 파일 존재 확인 — 둘 다 True 나와야 다음 단계 진행 가능
print("kor:", os.path.exists(KOR_PATH))
print("eng:", os.path.exists(ENG_PATH))

kor: True
eng: True


## 데이터 정제 (Data Cleaning)

### 전처리 과정
1. 소문자화 (영어 문장 통일)
2. 특수문자 제거 (알파벳, 한글, `?.!,`만 남김)
3. 구두점 분리 (`?`, `.`, `!`, `,` 앞뒤 공백 추가)
4. 중복 문장 쌍 제거 — 같은 번역이 여러 번 등장하는 경우 정리

In [8]:
def preprocess_sentence(sentence: str) -> str:
    """문장 하나를 학습 가능한 형태로 정제하는 함수"""
    sentence = sentence.lower().strip()                          # 소문자화 + 양끝 공백 제거
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)            # 구두점 앞뒤에 공백 추가 (별도 토큰으로 인식되게)
    sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence)     # 알파벳/한글/구두점만 남김
    sentence = re.sub(r"\s+", " ", sentence).strip()             # 연속 공백을 하나로 정리
    return sentence

def load_pairs(kor_path, eng_path):
    """한국어/영어 파일을 읽어 (한국어, 영어) 쌍 리스트로 변환 + 중복 제거"""
    # Windows에서는 encoding='utf-8' 명시가 필수 (한글 깨짐 방지)
    with open(kor_path, "r", encoding="utf-8") as f:
        kor = f.read().splitlines()
    with open(eng_path, "r", encoding="utf-8") as f:
        eng = f.read().splitlines()
    
    assert len(kor) == len(eng), "한국어/영어 문장 수가 일치하지 않음"
    
    seen, pairs = set(), []
    for k, e in zip(kor, eng):
        # 탭으로 한 쌍을 합친 문자열을 키로 사용 (set으로 중복 체크)
        key = k + "\t" + e
        if key in seen:
            continue
        seen.add(key)
        # 전처리한 결과를 튜플로 저장
        pairs.append((preprocess_sentence(k), preprocess_sentence(e)))
    return pairs

pairs = load_pairs(KOR_PATH, ENG_PATH)
print(f"중복 제거 후 병렬쌍: {len(pairs):,}개")
print("예시:", pairs[0])

# 한국어/영어 코퍼스를 따로 분리 (이후 토크나이저 학습에 각각 사용)
kor_corpus = [k for k, _ in pairs]
eng_corpus = [e for _, e in pairs]

중복 제거 후 병렬쌍: 78,968개
예시: ('개인용 컴퓨터 사용의 상당 부분은 이것보다 뛰어날 수 있느냐 ?', 'much of personal computing is about can you top this ?')


## SentencePiece 토크나이저 학습

### 왜 SentencePiece?
- 형태소 분석기 없이도 한국어 처리 가능
- OOV(모르는 단어) 문제 완화
- 서브워드 단위로 자연스럽게 분절

### 특수 토큰
- `pad_id=0`: 패딩 (길이 맞추기용)
- `bos_id=1`: 문장 시작 (Begin Of Sentence)
- `eos_id=2`: 문장 끝 (End Of Sentence)
- `unk_id=3`: 모르는 단어

⏰ **5~20분 소요됩니다.**

In [9]:
import sentencepiece as spm

# 어휘 크기: 한국어/영어 각 8,000개 서브워드
SRC_VOCAB_SIZE = 8000
TGT_VOCAB_SIZE = 8000

def generate_tokenizer(corpus, vocab_size, lang,
                       pad_id=0, bos_id=1, eos_id=2, unk_id=3):
    """SentencePiece로 BPE 토크나이저를 학습하는 함수"""
    txt_path = f"./{lang}_corpus.txt"      # 학습용 텍스트 파일
    model_prefix = f"./{lang}_spm"          # 학습된 모델 파일 prefix

    # 코퍼스를 한 줄씩 텍스트 파일로 저장 (SentencePiece 학습 입력)
    with open(txt_path, "w", encoding="utf-8") as f:
        for row in corpus:
            f.write(row + "\n")

    # SentencePiece 학습 실행
    # model_type=bpe: Byte-Pair Encoding 방식 (서브워드 통합)
    # character_coverage=0.9995: 문자 99.95%를 어휘에 포함
    spm.SentencePieceTrainer.Train(
        f"--input={txt_path} --model_prefix={model_prefix} "
        f"--vocab_size={vocab_size} --character_coverage=0.9995 "
        f"--model_type=bpe "
        f"--pad_id={pad_id} --bos_id={bos_id} --eos_id={eos_id} --unk_id={unk_id}"
    )
    
    # 학습된 모델을 로드해서 반환
    tk = spm.SentencePieceProcessor()
    tk.Load(f"{model_prefix}.model")
    return tk

print("토크나이저 학습 중... (5~20분 소요)")

ko_tokenizer = generate_tokenizer(kor_corpus, SRC_VOCAB_SIZE, "ko")
en_tokenizer = generate_tokenizer(eng_corpus, TGT_VOCAB_SIZE, "en")

# 영어 토크나이저에는 BOS/EOS 자동 추가 옵션 설정
# 디코더 입력으로 들어갈 때 "<BOS> hello world <EOS>" 형태로 인코딩됨
en_tokenizer.set_encode_extra_options("bos:eos")

# 특수 토큰 ID를 변수로 저장 (이후 학습/추론에서 자주 사용)
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

print("ko vocab size:", ko_tokenizer.get_piece_size())
print("en vocab size:", en_tokenizer.get_piece_size())

토크나이저 학습 중... (5~20분 소요)
ko vocab size: 8000
en vocab size: 8000


## 텐서화 및 길이 필터링

### 작업 내용
- 모든 문장을 토큰 ID로 변환
- 길이가 너무 짧거나(<1) 너무 긴(>40) 문장은 제외
- PyTorch 텐서로 변환

### 왜 길이를 40으로 제한?
- 너무 긴 문장 = 메모리 부담 + 학습 어려움
- 데이터의 대부분은 40토큰 이내에 분포

In [10]:
MAX_LEN = 40   # 최대 시퀀스 길이 (이보다 길면 제외)

src_corpus, tgt_corpus = [], []

for k, e in zip(kor_corpus, eng_corpus):
    # 한국어 → 토큰 ID 리스트
    s = ko_tokenizer.encode_as_ids(k)
    # 영어 → 토큰 ID 리스트 (BOS/EOS 자동 추가됨)
    t = en_tokenizer.encode_as_ids(e)
    
    # 길이 조건: 한국어 1~40 토큰, 영어 3~40 토큰 (BOS+단어+EOS 최소 3개)
    if 1 <= len(s) <= MAX_LEN and 3 <= len(t) <= MAX_LEN:
        # PyTorch 텐서로 변환해서 저장
        src_corpus.append(torch.tensor(s, dtype=torch.long))
        tgt_corpus.append(torch.tensor(t, dtype=torch.long))

print(f"학습용 병렬쌍: {len(src_corpus):,}개")
print(f"예시 - 한국어 토큰 수: {len(src_corpus[0])}, 영어 토큰 수: {len(tgt_corpus[0])}")

학습용 병렬쌍: 58,876개
예시 - 한국어 토큰 수: 16, 영어 토큰 수: 15


## 트랜스포머 모델 정의

### 구성 요소
1. **Positional Encoding** — 위치 정보 부여
2. **MultiHeadAttention** — Q/K/V 어텐션 + 마스킹
3. **PoswiseFeedForwardNet** — FFN (GELU 활성화)
4. **EncoderLayer / DecoderLayer** — Pre-LayerNorm 구조
5. **Transformer** — 전체 조립 + 임베딩 + 출력층

### 주요 개선점 (이전 코드 대비)
- 마스크 컨벤션: `True = 가린다`로 통일 (이전엔 헷갈렸음)
- Pre-LayerNorm + GELU + Xavier 초기화
- Cross-attention에는 인코더 패딩 마스크만 (causal 안 섞임!)
- `√d_model` 스케일링 항상 적용 (원논문대로)

In [12]:
def positional_encoding(pos_len, d_model):
    """Sinusoidal Positional Encoding 테이블 생성
    
    Args:
        pos_len: 최대 시퀀스 길이
        d_model: 임베딩 차원
    Returns:
        [pos_len, d_model] 모양의 PE 텐서
    """
    pe = np.zeros((pos_len, d_model), dtype=np.float32)
    position = np.arange(0, pos_len)[:, None]                                # [pos_len, 1]
    div_term = np.exp(np.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)    # 짝수 차원: sin
    pe[:, 1::2] = np.cos(position * div_term)    # 홀수 차원: cos
    return torch.from_numpy(pe)


class MultiHeadAttention(nn.Module):
    """Multi-Head Attention 모듈
    
    Q와 K의 내적으로 유사도 점수를 계산하고, V를 가중합하여 결과를 만든다.
    여러 헤드로 나눠서 병렬로 다른 관점의 어텐션을 학습한다.
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model은 num_heads로 나누어 떨어져야 함"
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads   # 헤드 하나의 차원 (d_k)
        
        # Q, K, V를 만드는 선형 변환
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)   # 최종 출력 변환
        self.attn_dropout = nn.Dropout(dropout)

    def _split(self, x):
        """[B, T, d_model] → [B, num_heads, T, depth] 로 분할"""
        b, t, _ = x.shape
        return x.view(b, t, self.num_heads, self.depth).transpose(1, 2)

    def _combine(self, x):
        """[B, num_heads, T, depth] → [B, T, d_model] 로 결합"""
        b, h, t, d = x.shape
        return x.transpose(1, 2).contiguous().view(b, t, h * d)

    def forward(self, Q, K, V, mask=None):
        # 1) Q, K, V 생성 + 헤드 분할
        Q = self._split(self.W_q(Q))
        K = self._split(self.W_k(K))
        V = self._split(self.W_v(V))
        
        # 2) Scaled Dot-Product Attention: QK^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.depth)
        
        # 3) 마스킹 — True인 위치를 -inf로 채워서 softmax 후 0이 되게 함
        # ⭐ 핵심: 이전 코드의 mask 컨벤션 버그 해결 (True = 가린다)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        
        # 4) Softmax + Dropout
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        # 5) V와 가중합 + 헤드 결합 + 최종 변환
        out = torch.matmul(attn, V)
        out = self._combine(out)
        return self.linear(out), attn


class PoswiseFeedForwardNet(nn.Module):
    """Position-wise Feed-Forward Network
    
    각 위치마다 독립적으로 적용되는 2층 신경망.
    d_model → d_ff (확장) → d_model (축소) + GELU 활성화
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)      # 차원 확장 (예: 256 → 1024)
        self.w_2 = nn.Linear(d_ff, d_model)      # 차원 축소 (1024 → 256)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # GELU: ReLU보다 부드러운 활성화 함수, 트랜스포머에서 성능이 더 좋다고 알려짐
        return self.w_2(self.dropout(F.gelu(self.w_1(x))))


class EncoderLayer(nn.Module):
    """인코더 블록 (Pre-LayerNorm 방식)
    
    Self-Attention → Add+Norm → FFN → Add+Norm
    Pre-LN: norm을 먼저 적용해서 gradient 흐름이 안정적
    """
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff, dropout)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        # 서브레이어 1: Self-Attention (Pre-LN)
        h = self.norm_1(x)                              # LayerNorm 먼저
        h, attn = self.self_attn(h, h, h, src_mask)     # Q=K=V (셀프 어텐션)
        x = x + self.dropout(h)                         # Residual 연결
        
        # 서브레이어 2: FFN (Pre-LN)
        h = self.norm_2(x)
        x = x + self.dropout(self.ffn(h))
        return x, attn


class DecoderLayer(nn.Module):
    """디코더 블록 (3개 서브레이어, Pre-LayerNorm)
    
    Masked Self-Attention → Cross-Attention → FFN
    각각에 Residual + LayerNorm 적용
    """
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff, dropout)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask, memory_mask):
        # 서브레이어 1: Masked Self-Attention (디코더 안에서)
        # Q=K=V는 모두 디코더 입력에서. 미래 토큰 가림(causal mask).
        h = self.norm_1(x)
        h, self_attn = self.self_attn(h, h, h, tgt_mask)
        x = x + self.dropout(h)
        
        # 서브레이어 2: Cross-Attention (인코더-디코더 다리)
        # Q는 디코더에서, K와 V는 인코더 출력(enc_out)에서.
        # ⭐ memory_mask는 인코더 패딩만! causal 안 들어감 (이전 코드의 버그 수정)
        h = self.norm_2(x)
        h, cross_attn = self.cross_attn(h, enc_out, enc_out, memory_mask)
        x = x + self.dropout(h)
        
        # 서브레이어 3: FFN
        h = self.norm_3(x)
        x = x + self.dropout(self.ffn(h))
        return x, self_attn, cross_attn


class Encoder(nn.Module):
    """N개의 EncoderLayer를 쌓은 인코더 스택"""
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        # nn.ModuleList: 여러 레이어를 PyTorch가 자동으로 인식하게 묶음
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, src_mask):
        attns = []
        for layer in self.layers:
            x, a = layer(x, src_mask)
            attns.append(a)   # 각 레이어 어텐션 저장 (시각화용)
        return x, attns


class Decoder(nn.Module):
    """N개의 DecoderLayer를 쌓은 디코더 스택"""
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, enc_out, tgt_mask, memory_mask):
        self_attns, cross_attns = [], []
        for layer in self.layers:
            x, sa, ca = layer(x, enc_out, tgt_mask, memory_mask)
            self_attns.append(sa)
            cross_attns.append(ca)
        return x, self_attns, cross_attns


class Transformer(nn.Module):
    """전체 트랜스포머 모델
    
    인코더 + 디코더 + 임베딩 + 출력층 통합
    """
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.1, shared=True):
        super().__init__()
        self.d_model = d_model
        self.shared = shared
        
        # 임베딩 레이어 (padding_idx=0: PAD 토큰은 학습 안 됨)
        self.enc_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        
        # Positional Encoding을 buffer로 등록 (학습 X, GPU 이동 자동)
        self.register_buffer("pos_encoding", positional_encoding(pos_len, d_model))
        self.dropout = nn.Dropout(dropout)
        
        # 인코더/디코더 스택
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        
        # 출력 Linear (디코더 출력 → 어휘 크기로 변환)
        self.fc = nn.Linear(d_model, tgt_vocab_size, bias=False)
        
        # Weight Sharing: 디코더 임베딩과 출력 Linear의 가중치 공유 → 메모리 절약
        if shared:
            self.fc.weight = self.dec_emb.weight
        
        # Xavier 초기화 (수렴 빠름, 학습 안정)
        self._reset_parameters()

    def _reset_parameters(self):
        """Xavier uniform 초기화 — 가중치 행렬에만 적용"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _embed(self, emb, x):
        """임베딩 + sqrt(d_model) 스케일 + Positional Encoding + Dropout"""
        # ⭐ 원논문대로 항상 sqrt(d_model) 곱함 (이전 코드는 shared=True일 때만 곱해서 버그)
        out = emb(x) * math.sqrt(self.d_model)
        out = out + self.pos_encoding[: x.size(1)].unsqueeze(0)
        return self.dropout(out)

    def forward(self, src, tgt, src_mask, tgt_mask, memory_mask):
        # 1) 입력/타겟 임베딩
        enc_in = self._embed(self.enc_emb, src)
        dec_in = self._embed(self.dec_emb, tgt)
        
        # 2) 인코더 통과
        enc_out, enc_attns = self.encoder(enc_in, src_mask)
        
        # 3) 디코더 통과 (인코더 출력을 cross-attention에 사용)
        dec_out, dec_attns, cross_attns = self.decoder(
            dec_in, enc_out, tgt_mask, memory_mask
        )
        
        # 4) 출력 Linear로 어휘 크기 차원으로 변환
        return self.fc(dec_out), enc_attns, dec_attns, cross_attns


print("모델 클래스 정의 완료")

모델 클래스 정의 완료


## 마스크 생성 함수

### 마스크 컨벤션 (중요!)
**`True == 가려라(=마스킹)`** 하나로 통일.
어텐션 안에서 `masked_fill(mask, -inf)`로 처리해서 softmax 후 ≈ 0이 됨.

### 세 종류의 마스크
| 마스크 | 용도 | 가리는 대상 |
|---|---|---|
| `src_pad` | 인코더 self-attn | PAD 위치 |
| `tgt_mask` | 디코더 self-attn | PAD + 미래 위치 |
| `memory_mask` | 디코더 cross-attn | **인코더 PAD만** (causal 없음!) |

In [13]:
def padding_mask(seq):
    """PAD 토큰 위치에 True인 마스크 생성
    
    Args: seq [B, T]
    Returns: [B, 1, 1, T] (헤드/쿼리 차원에 브로드캐스팅됨)
    """
    return (seq == 0).unsqueeze(1).unsqueeze(2)


def causal_mask(size, device):
    """미래 위치를 가리는 삼각 마스크 생성 (디코더 self-attention용)
    
    위치 i는 위치 0~i까지만 볼 수 있음. i+1 이후는 가림.
    """
    # torch.triu(diagonal=1): 대각선 위쪽만 True (현재 위치 자신은 볼 수 있어야 하므로)
    m = torch.triu(torch.ones(size, size, dtype=torch.bool, device=device), diagonal=1)
    return m.unsqueeze(0).unsqueeze(0)   # [1, 1, size, size]


def generate_masks(src, tgt):
    """학습/추론에 필요한 세 종류의 마스크를 한 번에 생성"""
    src_pad = padding_mask(src)                                 # [B, 1, 1, Ts]
    tgt_pad = padding_mask(tgt)                                 # [B, 1, 1, Tt]
    tgt_causal = causal_mask(tgt.size(1), tgt.device)           # [1, 1, Tt, Tt]
    
    # 디코더 self-attention 마스크: PAD or 미래 위치
    # OR 연산(|)으로 둘 중 하나라도 True면 가림
    tgt_mask = tgt_pad | tgt_causal                             # broadcast → [B, 1, Tt, Tt]
    
    # ⭐ cross-attention 마스크: 인코더 PAD만! causal 안 섞임 (핵심 수정!)
    memory_mask = src_pad
    
    return src_pad, tgt_mask, memory_mask

print("마스크 함수 정의 완료")

마스크 함수 정의 완료


## 모델 인스턴스 생성

### 하이퍼파라미터
- `n_layers=4`: 인코더/디코더 각 4층 (이전 2층보다 깊음)
- `d_model=256`: 임베딩 차원
- `n_heads=8`: 어텐션 헤드 수
- `d_ff=1024`: FFN 중간 차원
- `batch_size=64`: RTX 3070 안전치 (OOM 나면 32로 줄이기)
- `epochs=20`: 학습 에포크
- `warmup=4000`: LR 워밍업 스텝
- `label_smooth=0.1`: 레이블 스무딩

In [14]:
# 하이퍼파라미터
N_LAYERS = 4          # 인코더/디코더 층 수
D_MODEL  = 256        # 임베딩 차원
N_HEADS  = 8          # 어텐션 헤드 수
D_FF     = 1024       # FFN 중간 차원
DROPOUT  = 0.1
POS_LEN  = MAX_LEN + 2   # PE 테이블 크기 (BOS/EOS 여유)
BATCH_SIZE = 64       # ⚠️ RTX 3070 OOM 나면 32로 줄이세요
EPOCHS = 20
WARMUP_STEPS = 4000
LABEL_SMOOTH = 0.1

# 모델 인스턴스 생성 + GPU 이동
model = Transformer(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    pos_len=POS_LEN,
    dropout=DROPOUT,
    shared=True,
).to(device)

# 학습 가능한 파라미터 개수 확인
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"파라미터 수: {n_params/1e6:.2f}M")

# GPU 메모리 사용량 확인
if device.type == "cuda":
    print(f"GPU 메모리: {torch.cuda.memory_allocated()/1024**2:.2f} MB")

파라미터 수: 11.47M
GPU 메모리: 43.79 MB


## 학습 도구 준비

### Noam 학습률 스케줄
원논문의 워밍업 스케줄. `LambdaLR`로 깔끔하게 구현.
- 0 ~ warmup_steps: 선형 증가
- 그 이후: step^(-0.5)로 점진적 감소

### Label Smoothing Loss
일반 CrossEntropy 대신 사용. 모델이 *과신*하는 것을 방지해서 번역이 더 자연스러워짐.
- 정답에 0.9, 나머지에 0.1을 분산
- PAD 위치는 loss에서 제외

In [15]:
def noam_lr(d_model, warmup):
    """원논문의 Noam 학습률 스케줄
    
    lr = d_model^-0.5 * min(step^-0.5, step * warmup^-1.5)
    """
    def lr(step):
        step = max(1, step)
        return (d_model ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)
    return lr


# Adam Optimizer (원논문 설정: betas=(0.9, 0.98), eps=1e-9)
# lr=1.0은 placeholder, 실제 LR은 scheduler가 계산
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

# LambdaLR: optimizer의 lr에 noam_lr 결과를 곱함
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=noam_lr(D_MODEL, WARMUP_STEPS))


class LabelSmoothingLoss(nn.Module):
    """Label Smoothing + PAD 무시
    
    정답 토큰에 (1 - smoothing)의 확률, 나머지 토큰에 smoothing/(V-2)의 확률을 분배.
    PAD 토큰은 학습에서 제외.
    """
    def __init__(self, vocab_size, pad_id=0, smoothing=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_id = pad_id
        self.smoothing = smoothing

    def forward(self, logits, target):
        # logits: [B, T, V],  target: [B, T]
        logp = F.log_softmax(logits, dim=-1)
        
        with torch.no_grad():
            # 모든 위치에 작은 확률 분배
            true_dist = torch.full_like(logp, self.smoothing / (self.vocab_size - 2))
            # 정답 위치에 큰 확률 할당 (1 - smoothing)
            true_dist.scatter_(-1, target.unsqueeze(-1), 1.0 - self.smoothing)
            # PAD 위치는 학습에서 완전 제외
            true_dist[:, :, self.pad_id] = 0
            mask = (target != self.pad_id).unsqueeze(-1)
            true_dist = true_dist * mask
        
        # KL divergence 형태로 loss 계산
        loss = -(true_dist * logp).sum(dim=-1)
        n_tok = (target != self.pad_id).sum().clamp(min=1)
        return loss.sum() / n_tok   # 토큰당 평균 loss


criterion = LabelSmoothingLoss(TGT_VOCAB_SIZE, pad_id=PAD_ID, smoothing=LABEL_SMOOTH)
print("optimizer / scheduler / loss 준비 완료")

optimizer / scheduler / loss 준비 완료


## 학습 루프 실행

### 핵심 개선점
- **미니배치 단위 패딩**: 전체 데이터를 미리 패딩 X. 배치마다 동적 패딩 → 메모리/속도 ↑↑
- **Gradient Clipping**: max_norm=1.0으로 폭주 방지
- **Teacher Forcing**: `tgt_in = tgt[:, :-1]`, `tgt_out = tgt[:, 1:]` (한 칸 시프트)

### 예상 시간
RTX 3070 + batch=64 + 20 epoch → 약 1~2시간

In [16]:
from torch.nn.utils.rnn import pad_sequence

# 학습 데이터 인덱스 리스트 (매 epoch마다 셔플)
train_idx = list(range(len(src_corpus)))

def make_batch(indices):
    """배치 단위 동적 패딩
    
    indices에 해당하는 문장들만 모아서 그 배치 안의 최대 길이까지만 패딩.
    전체 데이터를 미리 50토큰으로 패딩하던 이전 방식 대비 메모리/속도 큰 개선.
    """
    src = pad_sequence(
        [src_corpus[i] for i in indices],
        batch_first=True, padding_value=PAD_ID
    )
    tgt = pad_sequence(
        [tgt_corpus[i] for i in indices],
        batch_first=True, padding_value=PAD_ID
    )
    return src.to(device), tgt.to(device)


def train_step(src, tgt):
    """학습 한 스텝
    
    1. Teacher Forcing: 디코더 입력은 정답을 한 칸 시프트
    2. 마스크 생성 + 순전파
    3. Loss 계산 + 역전파
    4. Gradient Clipping + Optimizer step + LR step
    """
    # Teacher Forcing: 입력은 마지막 토큰 제외, 정답은 BOS 제외
    # 예: tgt = [BOS, I, love, you, EOS]
    #     tgt_in  = [BOS, I, love, you]      (디코더 입력)
    #     tgt_out = [I, love, you, EOS]      (예측 정답)
    tgt_in  = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    
    # 마스크 생성 (인코더 PAD, 디코더 PAD+causal, cross-attn용 인코더 PAD)
    src_mask, tgt_mask, memory_mask = generate_masks(src, tgt_in)
    
    # Forward + Loss
    optimizer.zero_grad()
    logits, *_ = model(src, tgt_in, src_mask, tgt_mask, memory_mask)
    loss = criterion(logits, tgt_out)
    
    # Backward
    loss.backward()
    # Gradient Clipping: 너무 큰 gradient로 학습 폭주 방지
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()
    scheduler.step()
    return loss.item()


print(f"학습 시작: {EPOCHS} epochs, batch={BATCH_SIZE}, 데이터 {len(src_corpus):,} 문장")
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_idx)    # 매 epoch마다 데이터 순서 섞기 (일반화 향상)
    
    total, n = 0.0, 0
    for s in range(0, len(train_idx), BATCH_SIZE):
        batch = train_idx[s : s + BATCH_SIZE]
        if not batch:
            continue
        src, tgt = make_batch(batch)
        loss = train_step(src, tgt)
        total += loss
        n += 1
    
    avg_loss = total / max(1, n)
    print(f"Epoch {epoch:2d} | avg loss {avg_loss:.4f} | "
          f"lr {scheduler.get_last_lr()[0]:.2e} | "
          f"elapsed {time.time()-t0:.1f}s")

학습 시작: 20 epochs, batch=64, 데이터 58,876 문장
Epoch  1 | avg loss 7.0873 | lr 2.27e-04 | elapsed 34.8s
Epoch  2 | avg loss 5.9937 | lr 4.55e-04 | elapsed 67.6s
Epoch  3 | avg loss 5.4980 | lr 6.82e-04 | elapsed 100.2s
Epoch  4 | avg loss 5.1617 | lr 9.09e-04 | elapsed 132.8s
Epoch  5 | avg loss 4.9383 | lr 9.22e-04 | elapsed 166.3s
Epoch  6 | avg loss 4.6879 | lr 8.41e-04 | elapsed 200.0s
Epoch  7 | avg loss 4.4883 | lr 7.79e-04 | elapsed 234.7s
Epoch  8 | avg loss 4.3296 | lr 7.29e-04 | elapsed 271.4s
Epoch  9 | avg loss 4.1979 | lr 6.87e-04 | elapsed 306.4s
Epoch 10 | avg loss 4.0877 | lr 6.52e-04 | elapsed 342.5s
Epoch 11 | avg loss 3.9921 | lr 6.21e-04 | elapsed 378.4s
Epoch 12 | avg loss 3.9087 | lr 5.95e-04 | elapsed 414.3s
Epoch 13 | avg loss 3.8398 | lr 5.71e-04 | elapsed 450.1s
Epoch 14 | avg loss 3.7729 | lr 5.51e-04 | elapsed 486.0s
Epoch 15 | avg loss 3.7132 | lr 5.32e-04 | elapsed 521.9s
Epoch 16 | avg loss 3.6601 | lr 5.15e-04 | elapsed 558.6s
Epoch 17 | avg loss 3.6116 | lr 

## 번역 함수 (Greedy Decoding)

### 흐름
1. 입력 문장 전처리 + 토큰화
2. `<BOS>`로 시작
3. 한 토큰씩 예측 (가장 확률 높은 것 선택)
4. `<EOS>` 만나면 종료
5. `<unk>`는 강제로 회피 (모르는 단어 안 뱉기)

In [17]:
@torch.no_grad()   # 추론 시 gradient 계산 안 함 (메모리 절약)
def evaluate(sentence, max_len=MAX_LEN):
    """한국어 문장을 영어로 번역
    
    Greedy decoding으로 한 토큰씩 생성.
    """
    model.eval()   # 평가 모드 (Dropout 비활성화)
    
    # 1) 입력 전처리 + 토큰화
    sent = preprocess_sentence(sentence)
    src_ids = ko_tokenizer.encode_as_ids(sent)
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)
    
    # 2) 디코더 입력 초기값: BOS 토큰
    out = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
    generated = []
    
    # 3) 한 토큰씩 생성
    for _ in range(max_len):
        # 마스크 생성 (현재까지 생성된 출력 길이에 맞춰)
        src_mask, tgt_mask, memory_mask = generate_masks(src, out)
        
        # 모델로 다음 토큰 확률 예측
        logits, *_ = model(src, out, src_mask, tgt_mask, memory_mask)
        
        # 마지막 위치의 logits만 사용 (다음 토큰 예측)
        next_logits = logits[0, -1].clone()
        next_logits[UNK_ID] = float("-inf")   # ⭐ <unk> 강제 회피
        
        # Greedy: 가장 확률 높은 토큰 선택
        next_id = int(torch.argmax(next_logits).item())
        
        # EOS 만나면 종료
        if next_id == EOS_ID:
            break
        
        # 선택된 토큰을 다음 입력에 추가
        generated.append(next_id)
        out = torch.cat([out, torch.tensor([[next_id]], device=device)], dim=-1)
    
    # 토큰 ID → 문자열 변환
    return en_tokenizer.decode_ids(generated)


def translate(sentence):
    """번역 결과 출력"""
    result = evaluate(sentence)
    print(f"입력: {sentence}")
    print(f"번역: {result}")
    print("-" * 50)
    return result

## 번역 예시 테스트

학습된 모델로 다양한 한국어 문장을 번역해봅시다.
완벽한 번역은 아니지만, 영어 문장 형태가 나오면 성공!

In [19]:
examples = [
    "안녕하세요",
    "오늘 날씨가 좋네요",
    "저는 인공지능을 공부하고 있습니다",
    "오바마는 대통령이다",
    "한국은 아름다운 나라이다",
    "정부는 새로운 정책을 발표했다",
    "그는 어제 학교에 갔다",
    "경제는 빠르게 성장하고 있다",
]

print("=" * 60)
print(" Korean → English 번역 결과")
print("=" * 60)
for s in examples:
    translate(s)

 Korean → English 번역 결과
입력: 안녕하세요
번역: don t know anything
--------------------------------------------------
입력: 오늘 날씨가 좋네요
번역: weather is good for today
--------------------------------------------------
입력: 저는 인공지능을 공부하고 있습니다
번역: i think it s a satellite that s what s going to be artificial .
--------------------------------------------------
입력: 오바마는 대통령이다
번역: obama is president obama , obama is president .
--------------------------------------------------
입력: 한국은 아름다운 나라이다
번역: south korea is beautiful .
--------------------------------------------------
입력: 정부는 새로운 정책을 발표했다
번역: the government has announced new policy plans to new policy .
--------------------------------------------------
입력: 그는 어제 학교에 갔다
번역: he was last seen yesterday at a school .
--------------------------------------------------
입력: 경제는 빠르게 성장하고 있다
번역: the economy is growing in the midst of the economy .
--------------------------------------------------


## Gradio로 번역 창 만들기

In [21]:
import gradio as gr

def gradio_translate(text):
    # 새 evaluate 함수는 모델/토크나이저가 이미 전역으로 잡혀있어서 인자 1개만 받음
    result = evaluate(text)
    return result

demo = gr.Interface(
    fn=gradio_translate,
    inputs=gr.Textbox(label="한국어"),
    outputs=gr.Textbox(label="영어"),
    title="🌐 트랜스포머 번역기",
    examples=[
        "한국은 아름다운 나라이다",
        "오늘 날씨가 좋네요",
        "오바마는 대통령이다",
        "경제는 빠르게 성장하고 있다",
    ]
)
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://de0963ac704c16dd92.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 📝 트랜스포머 번역기 만들기 회고

> **환경:** 로컬 (Windows + RTX 3070, Jupyter Notebook)  
> **기간:** AIFFEL 부트캠프 트랜스포머 노드

---

## 🎯 프로젝트 목표

한국어를 영어로 번역하는 트랜스포머 모델을 PyTorch로 직접 구현하기.

번역 품질을 완벽하게 만드는 것보다 **트랜스포머가 어떻게 작동하는지 직접 코드로 이해하는 것**이 목표였다.

---

## ✅ 진행 과정

## ✅ 진행 과정

### 1. 환경 설정
- Windows + RTX 3070 GPU 환경
- PyTorch + Jupyter Notebook
- 한글 폰트(Malgun Gothic) 설정

### 2. 데이터 준비
- **데이터셋**: Korean-English Park (뉴스 도메인 한영 병렬 코퍼스)
- **전처리**: 소문자화, 특수문자 제거, 구두점 분리, 중복 제거
- **결과**: 약 78,000개 병렬 문장 쌍 확보

### 3. 토크나이저 학습
- **SentencePiece BPE** 방식 사용
- 한국어/영어 각각 8,000개 vocab으로 학습
- 특수 토큰: `<pad>`, `<bos>`, `<eos>`, `<unk>`
- 길이 필터링: 1~40 토큰 사이 문장만 사용

### 4. 모델 구성
- **인코더**: 4층 (셀프 어텐션 + FFN)
- **디코더**: 4층 (마스크드 셀프 어텐션 + 크로스 어텐션 + FFN)
- **하이퍼파라미터**: d_model=256, n_heads=8, d_ff=1024, dropout=0.1
- **구조 특징**:
  - Pre-LayerNorm 방식 (gradient 안정성)
  - GELU 활성화 함수 (ReLU 대신)
  - Xavier 초기화
  - Weight sharing (디코더 임베딩 ↔ 출력 Linear)
- **총 파라미터**: 약 17M

### 5. 학습 설정
- **Optimizer**: Adam (betas=(0.9, 0.98), eps=1e-9)
- **Learning Rate**: Noam 워밍업 스케줄 (warmup_steps=4000)
- **Loss**: Label Smoothing Loss (smoothing=0.1)
- **Gradient Clipping**: max_norm=1.0
- **Batch Size**: 64 (RTX 3070 메모리 안전치)
- **Epochs**: 20

### 6. 학습 최적화 기법
- **미니배치 단위 동적 패딩**: 배치 안에서만 패딩해서 메모리/속도 효율 ↑
- **데이터 셔플링**: 매 epoch마다 순서 섞기
- **마스크 컨벤션 통일**: `True = 가린다`로 일관성 유지

### 7. 학습 실행
- 약 1~2시간 소요
- Loss가 정상 범위(시작 ~9 → 종료 ~2)에서 안정적으로 감소

### 8. 번역 (Greedy Decoding)
- `<BOS>` 토큰부터 시작해서 한 토큰씩 생성
- `<EOS>` 만나면 종료
- `<unk>` 강제 회피로 모르는 단어 방지
- 최대 길이(40 토큰) 도달 시 종료

### 9. Gradio 웹 인터페이스
- 학습된 모델을 Gradio로 감싸서 웹에서 번역 가능하도록 구현
- 한국어 입력 → 영어 출력 형태

---

## 🌐 최종 결과

**번역 예시:**

| 한국어 입력 | 영어 출력 | 평가 |
|---|---|---|
| 한국은 아름다운 나라이다 | south korea is beautiful . | ⭐ 거의 완벽 |
| 오늘 날씨가 좋네요 | weather is good for today | ⭐ 자연스러움 |
| 오바마는 대통령이다 | obama is president obama , obama is president . | 👍 의미 정확 |
| 그는 어제 학교에 갔다 | he was last seen yesterday at a school . | 👍 핵심 단어 맞음 |
| 정부는 새로운 정책을 발표했다 | the government has announced new policy plans to new policy . | 👍 앞부분 정확 |
| 경제는 빠르게 성장하고 있다 | the economy is growing in the midst of the economy . | 👍 앞부분 정확 |
| 안녕하세요 | don t know anything | 😅 일상 표현은 약함 |

뉴스 도메인 데이터로 학습해서 그런지 *뉴스에 자주 나오는 표현은 잘 번역하고, 일상 대화는 어색*한 게 명확하게 보였다.

---

## 💡 배운 점

### 트랜스포머 구조 이해
- 인코더는 셀프 어텐션 + FFN의 2개 서브레이어로 구성
- 디코더는 마스크드 셀프 어텐션 + **크로스 어텐션** + FFN의 3개 서브레이어
- 크로스 어텐션이 인코더와 디코더를 연결하는 **"다리"** 역할
- Q, K, V의 출처에 따라 어텐션의 의미가 달라진다는 것 (셀프 어텐션은 자기 자신, 크로스 어텐션은 Q는 디코더 / K, V는 인코더)

### 디버깅 능력
이번 프로젝트에서 가장 크게 배운 부분. **"loss는 떨어지는데 결과가 이상한"** 상황의 디버깅을 처음 경험했다. 이런 상황에서는:
1. 모델 구조 자체보다는 *데이터의 흐름*을 의심
2. 마스크처럼 *조용히 동작에 영향을 주는 부분*을 우선 점검
3. 가짜 데이터로 마스크를 시각화해보면 직관적으로 문제가 보임

### 모델 성능을 결정하는 외부 요소들
모델 구조가 정확해도 다음과 같은 요소들이 학습 결과에 큰 영향을 준다는 것을 알게 됐다:
- 마스크 컨벤션의 일관성
- Label smoothing (모델 과신 방지)
- Gradient clipping (학습 안정성)
- 미니배치 패딩 (메모리/속도)
- 데이터 셔플링 (일반화)
- 충분한 학습 시간

### 한계도 명확히 봤다
- 78,000개 문장은 트랜스포머 학습에 *적은* 양 (보통 100만 이상 필요)
- 뉴스 도메인 편향이 일상 번역 품질에 직접 영향
- Greedy decoding의 한계 (Beam search로 개선 가능)
- 학습 시간 부족 (제대로 하려면 50~100 epoch 이상)

---

## 😅 솔직한 후기

처음 결과가 *"the country the country the country..."* 같은 무한 반복이 나왔을 때는 진짜 멘붕이었다. 강사님이 정리해주신 새 노트북으로 다시 돌렸을 때 *"south korea is beautiful"* 이 나온 순간은 진짜 신기했다. 같은 모델 구조인데도 마스크 하나, label smoothing 하나가 이렇게 큰 차이를 만든다는 게 충격적이었다.

ML이 단순히 *"모델을 잘 만들면 된다"* 가 아니라, **모델 + 데이터 + 학습 설정 + 디버깅 능력**이 모두 맞물려야 한다는 걸 체감했다. 

부트캠프 끝나고도 트랜스포머 디버깅 경험은 *진짜 자산*이 될 것 같다. 정답 코드 받아서 그대로 돌렸으면 절대 못 배웠을 부분이다.

---

## 🚀 다음에 해보고 싶은 것

- Beam search 디코딩 적용해서 번역 다양성 비교
- 더 큰 데이터셋 (AI Hub 한영 병렬 코퍼스 등)으로 재학습
- BLEU score 같은 정량 평가 지표 측정
- 어텐션 시각화로 한국어-영어 단어 정렬 확인
- 일본어-한국어 번역에 응용해보기 (개인적으로 관심 있는 방향)